# NB12 — RQ2 Pseudo-Labels, Quality Score, Freeze U'

Produces the common RQ2 pool **U'** from NB11's frozen `U_clean`:
fixed RQ1 C0 teacher pseudo-labels (XLS-R CTC → BARTpho), interpretable confidence
features, one hard-eligibility rule, and a quality score calibrated on **G_validation only**.

## 0. Purpose and scientific guardrails
- **Input**: only the NB11 generation that `artifacts/rq2/u_clean/CURRENT` names, with `COMPLETE.json`
  and status `SUCCESS_RQ2_U_CLEAN_FROZEN`. Every segment WAV is re-hashed before inference.
- **Teacher**: the fixed RQ1 C0 chain proven by `rq1_final_contract.json` + `checkpoint_proof`.
  No checkpoint chosen by mtime, filename order, "latest", or test score. C0 vs D0 is never re-evaluated.
- **D0** (optional) is an agreement feature only, never a second teacher; if enabled it applies to the
  whole pool and to G_validation under one frozen contract.
- **Inference only**: `model.eval()`, `torch.inference_mode()`, greedy CTC / deterministic beam search.
  No training, no sampling. No Vietnamese reference is ever read for U.
- **Frozen G_test is never used** — not for teacher choice, normalisation, calibration, hard filters,
  thresholds, weights, model selection, or any other RQ2 decision. NB12 does not open it.
- **G_validation** human references are used only inside the isolated calibration function.
- **One U'** for every later treatment. Eligibility never reads the quality score.
- **Not in NB12**: N-hour Random/Top-Q selection, D-Random/D-Quality training, final G_test evaluation,
  paired bootstrap.
- `SUCCESS_RQ2_PSEUDO_POOL_FROZEN` + `CURRENT` exist only after a reviewer freezes the quality-score
  proposal by its hash. Before that the best possible status is `READY_FOR_RQ2_QUALITY_REVIEW`.

Code lives in `src/rq2_pseudo_contract.py`, `src/rq2_pseudo_label.py`, `src/rq2_quality.py`.

## 1. Imports and runtime paths

In [ ]:
import gc
import json
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / "requirements.txt").is_file() and (cand / "src").is_dir() and (cand / "notebooks").is_dir():
            return cand
    raise RuntimeError("cannot locate project root")

PROJECT_ROOT = _find_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml

from src.rq2_pseudo_contract import (
    PSEUDO_RELATIVE_DIR, DataAccessLedger, Nb11InputError, TeacherContractError,
    assert_nb11_input_unchanged, assert_nb12_output_dir, read_current_generation_id,
    resolve_fixed_teacher, resolve_nb11_input, verify_generation, verify_nb11_segment_audio,
)
from src.rq2_pseudo_label import (
    HfCtcAsrAdapter, HfDirectD0Adapter, HfSeq2SeqMtAdapter, InferenceCheckpoint,
    agreement_features, configure_deterministic_torch, ctc_greedy_confidence, inference_binding,
    items_from_audio_frame, items_from_nb11, iter_raw_records, materialize_validation_audio,
    run_peak_safe_teacher_inference, validation_audio_identity_sha256, validation_source_rows,
)
from src.rq2_quality import (
    QualityCalibrationConfig, build_quality_score_contract, calibrate_quality_score, derive_status,
    freeze_quality_score_contract, hard_eligibility, load_g_validation, nb12_source_has_no_selection,
    OutputSanityConfig, publish_u_prime_generation, write_review_bundle, GATE_NAMES,
)

CFG = yaml.safe_load((PROJECT_ROOT / "configs" / "rq1.yaml").read_text(encoding="utf-8"))
OUT_DIR = assert_nb12_output_dir(PROJECT_ROOT / PSEUDO_RELATIVE_DIR, PROJECT_ROOT)
MANIFESTS_DIR = PROJECT_ROOT / "data" / "manifests"
NB12_SOURCES = [PROJECT_ROOT / "src" / n for n in ("rq2_pseudo_contract.py", "rq2_pseudo_label.py", "rq2_quality.py")]
print("PROJECT_ROOT:", PROJECT_ROOT)
print("NB12 output dir:", OUT_DIR.relative_to(PROJECT_ROOT))

## 2. Flags (operator controls)

`RUN_FULL_PIPELINE` and `QUALITY_SCORE_FROZEN` stay `False` until NB11 has published its final frozen
`U_clean` **and** the code has been reviewed. `QUALITY_SCORE_FROZEN = True` additionally requires
`REVIEWED_QUALITY_PROPOSAL_SHA256` to equal the proposal hash printed in section 7 after human review.
The RQ1 inputs are exact identifiers, never "latest".

In [ ]:
RUN_FULL_PIPELINE = False
QUALITY_SCORE_FROZEN = False
REVIEWED_QUALITY_PROPOSAL_SHA256 = ""

D0_AGREEMENT_ENABLED = False          # whole pool + G_validation under one contract, or not at all

RQ1_FINAL_CONTRACT_HASH = ""          # full 64-hex rq1_final_contract_hash from Notebook 06
ASR_STATE_DIR = ""                    # exact NB03/NB04/NB05 state dirs that Notebook 06 used
MT_STATE_DIR = ""
DIRECT_STATE_DIR = ""
DEVICE = "cuda"
SHARD_SIZE = 256

CALIBRATION_CONFIG = QualityCalibrationConfig(
    weight_grid_step=0.1,
    normalization_lower_quantile=0.05,
    normalization_upper_quantile=0.95,
    penalties=(("mt_source_truncated", 0.0), ("mt_hit_max_length", 0.0)),
    min_calibration_records=200,
    output_sanity_enabled=False,
)

print("RUN_FULL_PIPELINE:", RUN_FULL_PIPELINE)
print("QUALITY_SCORE_FROZEN:", QUALITY_SCORE_FROZEN)
print("D0_AGREEMENT_ENABLED:", D0_AGREEMENT_ENABLED)
if QUALITY_SCORE_FROZEN and len(REVIEWED_QUALITY_PROPOSAL_SHA256) != 64:
    raise RuntimeError("QUALITY_SCORE_FROZEN needs the reviewed 64-hex quality_score_proposal_sha256")

## 3. Resolve and validate the NB11 input

Read-only. Refuses a missing/incomplete/unsuccessful generation, pin drift, schema drift,
duplicate UIDs, or missing audio. NB11's checkpoint directory is never read.

In [ ]:
NB11 = None
NB11_ERROR = None
try:
    NB11 = resolve_nb11_input(PROJECT_ROOT)
except Nb11InputError as exc:
    NB11_ERROR = str(exc)
NB11_INPUT_READY = NB11 is not None

print("NB11_INPUT_READY:", NB11_INPUT_READY)
if NB11_INPUT_READY:
    print("  generation:", NB11.generation_id)
    print("  nb11_input_contract_sha256:", NB11.contract_sha256)
    print("  segments:", NB11.contract["n_segments"], "| hours:", round(NB11.contract["total_duration_seconds"] / 3600, 3))
else:
    print("  reason:", NB11_ERROR)
if RUN_FULL_PIPELINE and not NB11_INPUT_READY:
    raise RuntimeError(f"refusing the real NB12 run: NB11 U_clean is not ready ({NB11_ERROR})")

## 4. Resolve the fixed C0 teacher (and optional D0 agreement)

Re-proves the RQ1 chain from the explicit final contract hash and state dirs: final-contract self-hash,
state-dir name, `SUCCESS_RQ1_FINAL`, source fingerprint, locked runtime, upstream handoffs,
checkpoint proof. Only identity fields of `rq1_final_summary.json` are kept (no test metrics).

In [ ]:
from src.rq1_contract import collect_runtime_provenance
from src.rq1_runtime_paths import resolve_rq1_runtime_paths

RUNTIME = resolve_rq1_runtime_paths(project_root=PROJECT_ROOT)
TEACHER = None
TEACHER_ERROR = None
if len(RQ1_FINAL_CONTRACT_HASH) == 64 and ASR_STATE_DIR and MT_STATE_DIR and DIRECT_STATE_DIR:
    try:
        TEACHER = resolve_fixed_teacher(
            project_root=PROJECT_ROOT,
            rq1_final_state_dir=RUNTIME.state_dir(RQ1_FINAL_CONTRACT_HASH),
            asr_state_dir=ASR_STATE_DIR,
            mt_state_dir=MT_STATE_DIR,
            direct_state_dir=DIRECT_STATE_DIR,
            d0_agreement_enabled=D0_AGREEMENT_ENABLED,
            actual_runtime=collect_runtime_provenance(),
        )
        if TEACHER["teacher_contract"]["rq1_final_contract_hash"] != RQ1_FINAL_CONTRACT_HASH:
            raise TeacherContractError("resolved RQ1 final contract differs from RQ1_FINAL_CONTRACT_HASH")
    except (TeacherContractError, RuntimeError, FileNotFoundError) as exc:
        TEACHER, TEACHER_ERROR = None, str(exc)
else:
    TEACHER_ERROR = "RQ1_FINAL_CONTRACT_HASH / ASR_STATE_DIR / MT_STATE_DIR / DIRECT_STATE_DIR not set"
TEACHER_CONTRACT_READY = TEACHER is not None

print("TEACHER_CONTRACT_READY:", TEACHER_CONTRACT_READY)
if TEACHER_CONTRACT_READY:
    print("  teacher_contract_sha256:", TEACHER["teacher_contract"]["teacher_contract_sha256"])
    print("  d0_agreement_contract_sha256:", TEACHER["d0_agreement_contract"]["d0_agreement_contract_sha256"])
else:
    print("  reason:", TEACHER_ERROR)
if RUN_FULL_PIPELINE and not TEACHER_CONTRACT_READY:
    raise RuntimeError(f"refusing the real NB12 run: teacher contract not proven ({TEACHER_ERROR})")

## 5. Build contracts

The decoding payload is part of the teacher hash. Resume checkpoints bind to the input identity,
teacher, D0, decoding, schema and code version; a mismatch fails closed.

In [ ]:
TEACHER_CONTRACT = D0_CONTRACT = DECODING = None
U_BINDING = None
if NB11_INPUT_READY and TEACHER_CONTRACT_READY:
    TEACHER_CONTRACT = TEACHER["teacher_contract"]
    D0_CONTRACT = TEACHER["d0_agreement_contract"]
    DECODING = dict(TEACHER_CONTRACT["decoding"])
    if D0_CONTRACT["enabled"]:
        DECODING["d0"] = D0_CONTRACT["decoding"]
    U_BINDING = inference_binding(
        pool="u_clean",
        input_identity_sha256=NB11.contract_sha256,
        teacher_contract_sha256=TEACHER_CONTRACT["teacher_contract_sha256"],
        d0_agreement_contract_sha256=D0_CONTRACT["d0_agreement_contract_sha256"],
        decoding_config_sha256=TEACHER_CONTRACT["decoding_config_sha256"],
    )
    print("decoding:", json.dumps(DECODING, sort_keys=True))
else:
    print("contracts: skipped (NB11 input or teacher not ready)")

## 6. Guarded raw pseudo-label inference (U_clean)

Peak-safe: ASR, then MT, then (optional) D0, one model resident at a time. Each restored checkpoint is
re-hashed against the RQ1 checkpoint proof before use. Shards are fixed-membership and resumable.

In [ ]:
def _release(restored, root, label):
    local_experiment_dir = restored.get("local_experiment_dir")

    def release():
        from src.rq1_restore import cleanup_local_rq1_experiment
        import torch

        restored.clear()
        cleanup_local_rq1_experiment(local_experiment_dir, allowed_root=root, label=label)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return release


def load_asr():
    from src.rq1_restore import assert_restored_checkpoint_matches_proof, restore_asr_for_rq1

    r = restore_asr_for_rq1(state_dir=ASR_STATE_DIR, local_ckpt_root=RUNTIME.local_asr_ckpt_root, device=DEVICE)
    assert_restored_checkpoint_matches_proof(r, TEACHER["checkpoint_proof"]["asr"], system="asr")
    return {"adapter": HfCtcAsrAdapter(r["model"], r["processor"]),
            "release": _release(r, RUNTIME.local_asr_ckpt_root, "asr")}


def load_mt():
    from src.rq1_restore import assert_restored_checkpoint_matches_proof, restore_mt_for_rq1

    r = restore_mt_for_rq1(state_dir=MT_STATE_DIR, local_ckpt_root=RUNTIME.local_mt_ckpt_root, device=DEVICE)
    assert_restored_checkpoint_matches_proof(r, TEACHER["checkpoint_proof"]["mt"], system="mt")
    mt = DECODING["mt"]
    adapter = HfSeq2SeqMtAdapter(r["model"], r["tokenizer"], max_source_length=mt["max_source_length"],
                                 generation_max_length=mt["generation_max_length"], num_beams=mt["num_beams"])
    return {"adapter": adapter, "release": _release(r, RUNTIME.local_mt_ckpt_root, "mt")}


def load_d0():
    from src.rq1_restore import assert_restored_checkpoint_matches_proof, restore_direct_for_rq1

    r = restore_direct_for_rq1(state_dir=DIRECT_STATE_DIR, local_ckpt_root=RUNTIME.local_direct_ckpt_root, device=DEVICE)
    assert_restored_checkpoint_matches_proof(r, TEACHER["checkpoint_proof"]["direct"], system="direct")
    d0 = DECODING["d0"]
    adapter = HfDirectD0Adapter(r["model"], r["feature_extractor"], r["tokenizer"], target_lang=d0["target_lang"],
                                generation_max_length=d0["generation_max_length"], num_beams=d0["num_beams"])
    return {"adapter": adapter, "release": _release(r, RUNTIME.local_direct_ckpt_root, "direct")}


U_CKPT = None
AUDIO_IDENTITY_VERIFIED = False
if RUN_FULL_PIPELINE:
    import torch

    if DEVICE != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("the real NB12 run needs CUDA")
    configure_deterministic_torch(int(CFG["seed"]))
    verify_nb11_segment_audio(NB11, progress_every=5000)
    AUDIO_IDENTITY_VERIFIED = True
    U_CKPT = InferenceCheckpoint(OUT_DIR / "checkpoint" / "u_clean", binding=U_BINDING,
                                 ordered_uids=NB11.ordered_uids, shard_size=SHARD_SIZE)
    U_REPORT = run_peak_safe_teacher_inference(
        ckpt=U_CKPT, items=items_from_nb11(NB11), decoding=DECODING,
        load_asr=load_asr, load_mt=load_mt, load_d0=load_d0 if D0_CONTRACT["enabled"] else None,
        progress=print,
    )
    print(json.dumps(U_REPORT, indent=2))
else:
    print("raw U_clean inference: skipped (RUN_FULL_PIPELINE=False)")

## 7. G_validation calibration (isolated)

Reads only `rq1_validation.csv` (hash-pinned by `split_summary.json`) through the data-access ledger.
Validation audio is a verified, resumable cache: a later run reuses a WAV only after the PCM hash
and source binding match, and that identity is the G_validation inference binding.
The teacher sees only G_validation audio; the human Vietnamese reference is joined only inside
`calibrate_quality_score`, which returns aggregates (normalisation bounds, all weight candidates,
Spearman ρ). Weights come from a pre-declared grid; nothing is tuned on U or G_test.

In [ ]:
LEDGER = DataAccessLedger()
CALIBRATION = QUALITY_CONTRACT = None
if RUN_FULL_PIPELINE:
    val_ids, VAL_REFS, val_identity = load_g_validation(MANIFESTS_DIR, ledger=LEDGER)
    val_dir = OUT_DIR / "g_validation"
    val_cache = materialize_validation_audio(
        val_ids, dataset_id=CFG["dataset"]["id"], parquet_revision=CFG["dataset"]["parquet_revision"],
        manifest_sha256=val_identity["manifest_sha256"],
        audio_dir=val_dir / "audio", parquet_cache_dir=val_dir / "_parquet_cache",
    )
    val_audio = val_cache["audio_frame"]
    val_identity["audio_identity_sha256"] = val_cache["validation_audio_identity_sha256"]
    print("validation audio cache:", {k: val_cache[k] for k in ("n_reused", "n_rebuilt", "n_invalidated", "cache_identity_sha256")})
    val_binding = inference_binding(
        pool="g_validation",
        input_identity_sha256=val_identity["audio_identity_sha256"],
        teacher_contract_sha256=TEACHER_CONTRACT["teacher_contract_sha256"],
        d0_agreement_contract_sha256=D0_CONTRACT["d0_agreement_contract_sha256"],
        decoding_config_sha256=TEACHER_CONTRACT["decoding_config_sha256"],
    )
    val_ckpt = InferenceCheckpoint(OUT_DIR / "checkpoint" / "g_validation", binding=val_binding,
                                   ordered_uids=val_audio["record_uid"].astype(str).tolist(), shard_size=SHARD_SIZE)
    run_peak_safe_teacher_inference(
        ckpt=val_ckpt, items=items_from_audio_frame(val_audio), decoding=DECODING,
        load_asr=load_asr, load_mt=load_mt, load_d0=load_d0 if D0_CONTRACT["enabled"] else None,
        progress=print,
    )
    val_identity["inference_binding_sha256"] = val_ckpt.binding_sha256
    val_records = list(iter_raw_records(
        val_ckpt, validation_source_rows(val_audio),
        identity={"teacher_contract_sha256": TEACHER_CONTRACT["teacher_contract_sha256"],
                  "d0_agreement_contract_sha256": D0_CONTRACT["d0_agreement_contract_sha256"]},
        d0_enabled=D0_CONTRACT["enabled"],
    ))
    CALIBRATION = calibrate_quality_score(val_records, VAL_REFS, d0_enabled=D0_CONTRACT["enabled"],
                                          config=CALIBRATION_CONFIG, calibration_identity=val_identity)
    del VAL_REFS
    QUALITY_CONTRACT = build_quality_score_contract(
        CALIBRATION,
        teacher_contract=TEACHER_CONTRACT,
        d0_agreement_contract=D0_CONTRACT,
        nb11_input_contract_sha256=NB11.contract_sha256,
    )
    print("quality_score_proposal_sha256:", QUALITY_CONTRACT["quality_score_proposal_sha256"])
    print("selected:", CALIBRATION["selected"])
    print("single-feature rho:", CALIBRATION["single_feature_spearman"])
    if QUALITY_SCORE_FROZEN:
        QUALITY_CONTRACT = freeze_quality_score_contract(
            QUALITY_CONTRACT, reviewed_proposal_sha256=REVIEWED_QUALITY_PROPOSAL_SHA256)
else:
    print("G_validation calibration: skipped (RUN_FULL_PIPELINE=False)")
print("QUALITY_SCORE_FROZEN:", bool(QUALITY_CONTRACT and QUALITY_CONTRACT["frozen"]))

## 8. Build U'

Hard eligibility first (never reads Q), then Q for eligible rows only. With an unfrozen quality
contract, every completed U raw record is scored and the U' gates are derived from that pass.
The result is a review bundle (`READY_FOR_RQ2_QUALITY_REVIEW` only when those gates pass).
It does not write a generation or `CURRENT`.

In [ ]:
PUBLISHED = REVIEW = None
if RUN_FULL_PIPELINE:
    assert_nb11_input_unchanged(NB11.contract, resolve_nb11_input(PROJECT_ROOT))
    UPSTREAM_GATES = {
        "nb11_input_valid": NB11_INPUT_READY,
        "nb11_input_unchanged": True,
        "teacher_contract_valid": TEACHER_CONTRACT_READY,
        "d0_agreement_contract_valid": bool(D0_CONTRACT) and D0_CONTRACT["enabled"] == D0_AGREEMENT_ENABLED,
        "audio_identity_verified": AUDIO_IDENTITY_VERIFIED,
        "no_g_test_access": not LEDGER.g_test_accessed,
        "no_reference_text_in_u_artifacts": True,
        "no_selection_or_training_in_nb12": nb12_source_has_no_selection(NB12_SOURCES),
    }
    identity = {"nb11_generation_id": NB11.generation_id, "nb11_input_contract_sha256": NB11.contract_sha256,
                "teacher_contract_sha256": TEACHER_CONTRACT["teacher_contract_sha256"],
                "d0_agreement_contract_sha256": D0_CONTRACT["d0_agreement_contract_sha256"]}
    raw_u = iter_raw_records(U_CKPT, NB11.rows, identity=identity, d0_enabled=D0_CONTRACT["enabled"])
    if QUALITY_CONTRACT["frozen"]:
        PUBLISHED = publish_u_prime_generation(
            OUT_DIR,
            raw_records=raw_u,
            expected_uids=NB11.ordered_uids, quality_contract=QUALITY_CONTRACT, calibration=CALIBRATION,
            teacher_contract=TEACHER_CONTRACT, d0_agreement_contract=D0_CONTRACT,
            nb11_input_contract=NB11.contract, ledger=LEDGER, upstream_gates=UPSTREAM_GATES,
        )
    else:
        REVIEW = write_review_bundle(
            OUT_DIR, raw_records=raw_u, expected_uids=NB11.ordered_uids,
            quality_contract=QUALITY_CONTRACT, calibration=CALIBRATION,
            teacher_contract=TEACHER_CONTRACT, d0_agreement_contract=D0_CONTRACT,
            nb11_input_contract=NB11.contract,
            ledger=LEDGER, upstream_gates=UPSTREAM_GATES,
        )
        print("review status:", REVIEW["summary"]["status"])
        print("prospective U' segments:", REVIEW["summary"]["pool"]["n_u_prime"])
        print("review bundle:", Path(REVIEW["review_dir"]).relative_to(PROJECT_ROOT))
else:
    print("U' build: skipped (RUN_FULL_PIPELINE=False)")

## 9. Validate / publish

In [ ]:
if PUBLISHED is not None:
    gen = read_current_generation_id(OUT_DIR)
    verified = verify_generation(OUT_DIR, gen)
    summary = PUBLISHED["summary"]
    print("status:", summary["status"], "| generation:", gen)
    print("U' segments:", summary["n_u_prime"], "| hours:", summary["u_prime_audio_hours"])
    print("exclusions:", summary["exclusions_by_reason"])
    print("artifacts:", sorted(verified["files"]))
else:
    print("publication: none (no frozen quality-score contract in this run)")

## 10. Offline synthetic checks and summary

Pure in-memory checks of the scoring rules; no model, no artifact read or written.

In [ ]:
import numpy as np

conf = ctc_greedy_confidence(np.log([[0.1, 0.9], [0.1, 0.9], [0.8, 0.2]]), [1, 1, 0], blank_id=0)
assert conf["asr_token_count"] == 1 and abs(conf["asr_confidence_raw"] - 0.9) < 1e-9
assert abs(agreement_features("xin chào", "xin chào")["teacher_d0_agreement_chrf"] - 100.0) < 1e-6
probe = {"asr_status": "OK", "asr_valid_encoding": True, "asr_empty": False, "asr_text_norm": "bah",
         "mt_status": "OK", "mt_valid_encoding": True, "mt_empty": False, "pseudo_vi_norm": "vi",
         "asr_mean_logprob": -0.1, "mt_mean_logprob": -0.2, "target_source_char_ratio": 0.67}
assert hard_eligibility(probe, d0_enabled=False, sanity=OutputSanityConfig()) == "U_PRIME_ELIGIBLE"
assert hard_eligibility({**probe, "mt_empty": True}, d0_enabled=False, sanity=OutputSanityConfig()) == "EXCLUDED_MT_EMPTY"
assert derive_status({g: True for g in GATE_NAMES} | {"quality_score_contract_frozen": False}) == "READY_FOR_RQ2_QUALITY_REVIEW"
assert nb12_source_has_no_selection(NB12_SOURCES)
print("offline synthetic checks: OK")

print("NB11_INPUT_READY:", NB11_INPUT_READY)
print("TEACHER_CONTRACT_READY:", TEACHER_CONTRACT_READY)
print("QUALITY_SCORE_FROZEN:", QUALITY_SCORE_FROZEN)
print("RUN_FULL_PIPELINE:", RUN_FULL_PIPELINE)